In [ ]:
import numpy as np

import trimesh
from trimesh.path.entities import Line
from trimesh.path import Path2D, Path3D

from math import tan, sin, cos, sqrt

import matplotlib.pyplot as plt

dt = 1.0
wheelbase = 0.5
cmds = [np.array([.8, np.random.choice([1, -1]) * .01])] * 20
step = 1

def move(x, dt, u, wheelbase):
    hdg = x[2]
    vel = u[0]
    steering_angle = u[1]
    dist = vel * dt
    if abs(steering_angle) > 0.001: # is robot turning?
        beta = (dist / wheelbase) * tan(steering_angle)
        r = wheelbase / tan(steering_angle) # radius
        sinh, sinhb = sin(hdg), sin(hdg + beta)
        cosh, coshb = cos(hdg), cos(hdg + beta)
        return x + np.array([-r*sinh + r*sinhb,
                              r*cosh - r*coshb, beta]), np.array([-vel*sinh + vel*sinhb, vel*cosh - vel*coshb]), hdg
    else: # moving in straight line
        return x + np.array([dist*cos(hdg), dist*sin(hdg), 0]), np.array([vel*cos(hdg), vel*sin(hdg)]), hdg


num_steps = 100
trajectories = np.empty((0,num_steps,3))
headings = np.empty((0,num_steps))
locations = []
spheres = []
for i in range(250):

    cmds = [np.array([np.random.choice([1, -1]) * np.random.uniform(.2, .8), np.random.choice([1, -1]) * .01])] * num_steps
    # Generate trajectories
    track = []
    velocity = []
    heading = []
    sim_pos = np.array([np.random.uniform(-500, 500), np.random.uniform(-500, 500), np.random.uniform(0, np.pi / 2)])
    # sim_pos = np.array([30, 5, 0])
    for j, u in enumerate(cmds):
        sim_pos, vel, hdg = move(sim_pos, dt/step, u, wheelbase)
        track_tmp = np.copy(sim_pos)
        track_tmp[2] = -10
        track.append(track_tmp)
        velocity.append(vel)
        heading.append(hdg)
    track = np.array(track)
    velocity = np.array(velocity)
    heading = np.array(heading)

    # plt.plot(track[:, 0], track[:,1], marker='.', color='k', lw=2)
    # plt.axis('equal')
    # plt.title("Robot  Trajectory")
    # plt.show()

    # mesh = trimesh.load_mesh("models/canyon.ply")
    # mesh = trimesh.load_mesh("models/lunar_mesh_ex.ply")
    mesh = trimesh.load_mesh("models/meshes_512/small_mesh0.ply")

    # create some rays
    ray_origins = track
    ray_directions = np.array([[0, 0, 1]]*track.shape[0])
    # ray_origins = np.array([[0, 0, -5], [2, 2, -10]])
    # ray_directions = np.array([[0, 0, 1]]*ray_origins.shape[0])

    # run the mesh- ray test
    locations, index_ray, index_tri = mesh.ray.intersects_location(
        ray_origins=ray_origins, ray_directions=ray_directions
    )

    if len(locations) < len(cmds) or locations.shape != (num_steps, 3):
        continue
    # stack rays into line segments for visualization as Path3D
    ray_visualize = trimesh.load_path(
        np.hstack((ray_origins[:1], ray_origins[:1] + ray_directions[:1])).reshape(-1, 2, 3)
    )
    # if locations.shape != (20, 3):
    #     import pdb; pdb.set_trace()
    
    locations = locations[np.argsort(index_ray)]
    locations[:, 2] += 0.5
    locations_reshaped = locations.reshape(1, num_steps, 3)
    trajectories = np.concatenate((trajectories, locations_reshaped), axis=0) 
    headings = np.concatenate((headings, heading.reshape(1, num_steps)), axis=0)       

    # ray_origins = np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]])
    # ray_directions = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
    # ray_visualize = trimesh.load_path(
    #     np.hstack((ray_origins, ray_origins + ray_directions * 5.0)).reshape(-1, 2, 3),
    #     colors=[(255, 0, 0, 255), (0, 255, 0, 255), (0, 0, 255, 255)]
    # )

radius = 1.0
for j in range(trajectories.shape[0]):
    for i in range(trajectories.shape[1]):
        sphere = trimesh.creation.icosphere(radius=radius, color=[1, 0, 0, 1])
        sphere.apply_translation(trajectories[j,i,:])
        spheres.append(sphere)
    # # create a visualization scene with rays, hits, and mesh
# scene = trimesh.Scene([mesh, ray_visualize, trimesh.points.PointCloud(locations)])

    # # # mesh.show()


import os
file_path = "small_mesh0.npy"
if os.path.exists(file_path):
    existing_data = np.load(file_path)  # Load existing data
    combined_data = np.concatenate((existing_data, trajectories))  # Append
else:
    combined_data = trajectories  # If file doesn't exist, just save new data

np.save(file_path, combined_data)
print(f'headings shape {headings.shape}')
np.save("headings.npy", heading)

print(f'trajectories {trajectories.shape}')
scene = trimesh.Scene([mesh, spheres])

scene.show()

In [ ]:
import numpy as np
import trimesh
data = np.load('small_mesh0.npy')[:10,:,:]
print(f"Data shape {data.shape}")

mesh = trimesh.load_mesh("models/meshes_512/small_mesh0.ply")

spheres = []
radius = 1.0
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        sphere = trimesh.creation.icosphere(radius=radius, color=[1, 0, 0, 1])
        sphere.apply_translation(data[i,j,:])
        spheres.append(sphere)

# scene = trimesh.Scene([mesh, spheres])
# scene.show()